In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.dataset import ImageDataset
from torch.utils.data import DataLoader

annotations_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annotations_file_trainval, img_dir_trainval)
trainval_dl = DataLoader(trainval_dataset, batch_size=32, shuffle=True)

In [3]:
from src.model import Model

model = Model()

In [6]:
from src.configs import (S, C, B, IDX_TO_CLASS, CONFIDENCE_THRESHOLD,
                         NMS_IOU_THRESHOLD)
from src.utils import convert_xywh_coords, IoU
from operator import itemgetter

def decode_preds(preds_batch):
    decoded_preds = []

    for pred in preds_batch:
        pred = pred.reshape((S, S, C + B * 5))
        objects = []

        for i in range(S):
            for j in range(S):
                pred_cell = pred[i][j]

                pred_class_idx = pred_cell[:20].argmax().item()
                pred_class_prob = pred_cell[pred_class_idx].item()
                pred_class = IDX_TO_CLASS[pred_class_idx]

                pred_1_confidence = (pred_cell[24].item() * \
                                     pred_class_prob,)
                pred_2_confidence = (pred_cell[29].item() * \
                                     pred_class_prob,)

                bbox_1 = convert_xywh_coords(pred_cell[20:24], i, j, False)
                bbox_2 = convert_xywh_coords(pred_cell[25:29], i, j, False)

                objects.append((pred_class,) + pred_1_confidence + bbox_1)
                objects.append((pred_class,) + pred_2_confidence + bbox_2)

        decoded_preds.append(objects)

    return decoded_preds

def filter_group_sort_preds(decoded_preds):
    sorted_preds = []

    # 1. filter and group remaining predictions by class
    for image in decoded_preds:
        valid_preds = {}
        for pred in image:
            if pred[1] > CONFIDENCE_THRESHOLD:
                class_name = pred[0]
                if class_name in valid_preds:
                    valid_preds[class_name].append(pred)
                else:
                    valid_preds[class_name] = [pred]

        sorted_preds.append(valid_preds)

    # 2. sort each class's predictions by confidence score
    for image in sorted_preds:
        for class_name in image:
            image[class_name].sort(key=itemgetter(1), reverse=True)

    return sorted_preds


def NMS(preds_batch):
    # 1. decode batch of predictions
    decoded_preds = decode_preds(preds_batch)

    # 2. filter, group, and sort the decoded predictions
    sorted_preds = filter_group_sort_preds(decoded_preds)

    # 3. perform Non-Maximum Suppression
    final_preds = []

    for image in sorted_preds:
        final_img_preds = {}

        for class_name, preds in image.items():
            final_img_preds[class_name] = []

            while preds:
                highest_conf = preds.pop(0)
                final_img_preds[class_name].append(highest_conf)

                preds = [pred for pred in preds if
                         IoU(highest_conf[2:6], pred[2:6]) < NMS_IOU_THRESHOLD]

        final_preds.append(final_img_preds)

    return final_preds

In [9]:
X_batch, y_batch = next(iter(trainval_dl))

preds = model(X_batch)
preds.shape

torch.Size([32, 1470])

In [11]:
decoded_preds = decode_preds(preds)
decoded_preds

[[('person',
   0.2930137996024307,
   tensor(-21.2485, grad_fn=<SubBackward0>),
   tensor(2.0221, grad_fn=<SubBackward0>),
   tensor(35.7950, grad_fn=<AddBackward0>),
   tensor(-0.0145, grad_fn=<AddBackward0>)),
  ('person',
   0.13644148795882494,
   tensor(32.9955, grad_fn=<SubBackward0>),
   tensor(-29.5805, grad_fn=<SubBackward0>),
   tensor(-40.2315, grad_fn=<AddBackward0>),
   tensor(4.7529, grad_fn=<AddBackward0>)),
  ('dog',
   -0.020587798285768066,
   tensor(20.2587, grad_fn=<SubBackward0>),
   tensor(-0.3999, grad_fn=<SubBackward0>),
   tensor(62.0374, grad_fn=<AddBackward0>),
   tensor(-7.1324, grad_fn=<AddBackward0>)),
  ('dog',
   -0.10808618160952577,
   tensor(55.7571, grad_fn=<SubBackward0>),
   tensor(4.9994, grad_fn=<SubBackward0>),
   tensor(24.5481, grad_fn=<AddBackward0>),
   tensor(8.1169, grad_fn=<AddBackward0>)),
  ('pottedplant',
   0.08546820033621927,
   tensor(41.4214, grad_fn=<SubBackward0>),
   tensor(13.0800, grad_fn=<SubBackward0>),
   tensor(72.7569, 

In [12]:
sorted_preds = filter_group_sort_preds(decoded_preds)
sorted_preds

[{},
 {'car': [('car',
    0.45928679233188774,
    tensor(16.9926, grad_fn=<SubBackward0>),
    tensor(140.3741, grad_fn=<SubBackward0>),
    tensor(-33.1464, grad_fn=<AddBackward0>),
    tensor(25.8070, grad_fn=<AddBackward0>))],
  'dog': [('dog',
    0.4311676889374496,
    tensor(159.9195, grad_fn=<SubBackward0>),
    tensor(96.5718, grad_fn=<SubBackward0>),
    tensor(142.3979, grad_fn=<AddBackward0>),
    tensor(104.6043, grad_fn=<AddBackward0>))],
  'bottle': [('bottle',
    0.7958623238066025,
    tensor(154.9090, grad_fn=<SubBackward0>),
    tensor(223.1847, grad_fn=<SubBackward0>),
    tensor(28.3087, grad_fn=<AddBackward0>),
    tensor(116.1631, grad_fn=<AddBackward0>))],
  'chair': [('chair',
    0.4757118494020496,
    tensor(179.4478, grad_fn=<SubBackward0>),
    tensor(157.6682, grad_fn=<SubBackward0>),
    tensor(139.7276, grad_fn=<AddBackward0>),
    tensor(189.8419, grad_fn=<AddBackward0>))],
  'boat': [('boat',
    0.430396524566401,
    tensor(176.1980, grad_fn=<Sub

In [14]:
final_preds = []

for image in sorted_preds:
    final_img_preds = {}

    for class_name, preds in image.items():
        final_img_preds[class_name] = []

        while preds:
            highest_conf = preds.pop(0)
            final_img_preds[class_name].append(highest_conf)

            preds = [pred for pred in preds if
                     IoU(highest_conf[2:6], pred[2:6]) < NMS_IOU_THRESHOLD]

    final_preds.append(final_img_preds)

In [15]:
final_preds

[{},
 {'car': [('car',
    0.45928679233188774,
    tensor(16.9926, grad_fn=<SubBackward0>),
    tensor(140.3741, grad_fn=<SubBackward0>),
    tensor(-33.1464, grad_fn=<AddBackward0>),
    tensor(25.8070, grad_fn=<AddBackward0>))],
  'dog': [('dog',
    0.4311676889374496,
    tensor(159.9195, grad_fn=<SubBackward0>),
    tensor(96.5718, grad_fn=<SubBackward0>),
    tensor(142.3979, grad_fn=<AddBackward0>),
    tensor(104.6043, grad_fn=<AddBackward0>))],
  'bottle': [('bottle',
    0.7958623238066025,
    tensor(154.9090, grad_fn=<SubBackward0>),
    tensor(223.1847, grad_fn=<SubBackward0>),
    tensor(28.3087, grad_fn=<AddBackward0>),
    tensor(116.1631, grad_fn=<AddBackward0>))],
  'chair': [('chair',
    0.4757118494020496,
    tensor(179.4478, grad_fn=<SubBackward0>),
    tensor(157.6682, grad_fn=<SubBackward0>),
    tensor(139.7276, grad_fn=<AddBackward0>),
    tensor(189.8419, grad_fn=<AddBackward0>))],
  'boat': [('boat',
    0.430396524566401,
    tensor(176.1980, grad_fn=<Sub